# 🏨 Hotel Reservation System 3: The Compliance Checker

The third agent screens every request against the hotel's rules before anyone acts on it. It retrieves the
relevant rules from a vector store of the hotel's rules (`hc.build_compliance_retriever()`, introduced in
`01_hotel_environment_and_coordinator.ipynb`) and returns a verdict (`COMPLIANT` or `VIOLATION`). The
coordinator then answers, forwards the booking, or refuses.

## Learning Objectives
1. **A RAG agent** - An LLM that must call `retrieve` before judging, with a `generate` node grounded in the results
2. **A three-agent graph** - Coordinator → compliance checker → coordinator → reservation agent
3. **A loop guard in routing** - Never send the same request to the compliance checker twice
4. **Checking verdicts in code** - Parse `COMPLIANT` / `VIOLATION` from the checker's message

## Where this fits
`02_hotel_reservation_agent.ipynb` added reservations. With compliance in place this is the **core system**
that notebooks 04–10 extend, starting with edge cases in `04_hotel_edge_cases_and_errors.ipynb`.

## Prerequisites
- The agents' prompts, printed in `01_hotel_environment_and_coordinator.ipynb` Part 3 (this notebook imports them from `hotel_common.py`)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` and imports the building blocks and `hotel_common`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from typing import Literal

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from datetime import date, timedelta

import hotel_common as hc  # shared hotel building blocks, in this folder (see 01_hotel_environment_and_coordinator.ipynb)
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: shared by all three agents
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🧰 Part 1: Data and Tools

In [ ]:
# ============================================================================
# DATABASE, SQL TOOLS AND THE COMPLIANCE RETRIEVER
# ============================================================================
db_path = hc.init_hotel_db("hotel.db")
sql_tools = hc.make_sql_tools(db_path)  # 02's tools: run_query only reads, book_room is the only write
retrieve = hc.build_compliance_retriever()
rag_tools = [retrieve]
print(f"✅ {len(sql_tools)} SQL tools, 1 retrieval tool over {len(hc.COMPLIANCE_RULES)} rules")

---
## 🛡️ Part 2: The Compliance Checker Nodes

Two nodes make up the checker:
- **`compliance_checker`** calls `retrieve` (bound as a tool) with the guest's request
- **`generate`** reads the retrieved rules and writes the verdict, grounded in exactly those rules. The retrieved
  text is passed into its prompt; a version that dropped the tool results would be judging from nothing.

The verdict message is tagged `name="compliance"` so code can find it later. The node that runs `generate` is
called `generate` too.

In [ ]:
# ============================================================================
# COMPLIANCE CHECKER AND GENERATE NODES
# ============================================================================
llm_with_rag_tools = llm.bind_tools(rag_tools)


def rag_assistant(state: MessagesState) -> dict:
    return {"messages": [llm_with_rag_tools.invoke([hc.COMPLIANCE_PROMPT] + state["messages"])]}


def generate(state: MessagesState) -> dict:
    """Write the verdict from the rules the retrieve tool just returned."""
    tool_messages = []
    for message in reversed(state["messages"]):
        if message.type != "tool":
            break
        tool_messages.insert(0, message)
    rules = "\n\n".join(m.content for m in tool_messages) or "(no rules retrieved)"
    last_request = next(m for m in reversed(state["messages"]) if m.type == "human")
    verdict = llm.invoke([
        hc.COMPLIANCE_PROMPT,
        SystemMessage(content=f"Retrieved rules:\n{rules}"),
        HumanMessage(content=f"Guest request: {last_request.content}\n\nGive your verdict."),
    ])
    return {"messages": [AIMessage(content=verdict.content, name="compliance")]}


print("✅ rag_assistant and generate defined")

---
## 🔀 Part 3: Routing With a Loop Guard

The coordinator is told to send **every** new request to the compliance checker first. Once a verdict exists for
the latest guest message, a reply that names the checker again would loop forever, so `choose_next_node` checks
that in code: it drops `compliance_checker` from the names it looks for. A reply such as "the compliance checker
approved it, forwarding to reservation_assistant" then still reaches the reservation agent.

These nodes and this router are the **core system** the later notebooks extend. `hotel_common.py` keeps the same
pieces for them: `hc.make_core_nodes()`, `hc.route_by_mention()` and the `hc.COMPLIANCE_FIRST` guideline.

In [ ]:
# ============================================================================
# COORDINATOR, ROUTER AND GRAPH
# ============================================================================
AGENTS = ["reservation_assistant", "compliance_checker"]
conversation_prompt = hc.conversation_prompt(
    AGENTS,
    extra_guidelines="- Forward every NEW guest request to compliance_checker first. If its verdict is VIOLATION, "
    "politely refuse and do not forward anywhere. If COMPLIANT, answer or forward as usual.",
)
llm_with_sql_tools = llm.bind_tools(sql_tools)


def conv_assistant(state: MessagesState) -> dict:
    return {"messages": [llm.invoke([conversation_prompt] + state["messages"])]}


def sql_assistant(state: MessagesState) -> dict:
    return {"messages": [llm_with_sql_tools.invoke([hc.SQL_PROMPT] + state["messages"])]}


def checked_since_last_guest_message(messages) -> bool:
    for m in reversed(messages):
        if m.type == "human":
            return False
        if getattr(m, "name", None) == "compliance":
            return True
    return False


def choose_next_node(state: MessagesState) -> Literal["reservation_assistant", "compliance_checker", "__end__"]:
    # Loop guard: once the latest request has a verdict, the checker is no longer a name to look for
    checked = checked_since_last_guest_message(state["messages"])
    candidates = [agent for agent in AGENTS if not (agent == "compliance_checker" and checked)]
    return hc.mentioned_agent(state["messages"][-1].content, candidates) or END


builder = StateGraph(MessagesState)
builder.add_node("conv_assistant", conv_assistant)
builder.add_node("reservation_assistant", sql_assistant)
builder.add_node("compliance_checker", rag_assistant)
builder.add_node("generate", generate)
builder.add_node("sql_tools", ToolNode(sql_tools))
builder.add_node("rag_tools", ToolNode(rag_tools))

builder.add_edge(START, "conv_assistant")
builder.add_conditional_edges("conv_assistant", choose_next_node, ["reservation_assistant", "compliance_checker", END])
builder.add_conditional_edges("reservation_assistant", tools_condition, {"tools": "sql_tools", END: END})
builder.add_conditional_edges("compliance_checker", tools_condition, {"tools": "rag_tools", END: "conv_assistant"})
builder.add_edge("sql_tools", "reservation_assistant")
builder.add_edge("rag_tools", "generate")
builder.add_edge("generate", "conv_assistant")

graph = builder.compile(checkpointer=InMemorySaver())
print("✅ Graph compiled. Edges:")
hc.print_edges(graph)

---
## ▶️ Part 4: Three Requests, Three Outcomes

- an ordinary question (should be `COMPLIANT` and answered)
- a request for another guest's dates (should be a `VIOLATION` of rule 4 and refused)
- a complete booking request (should be `COMPLIANT` and forwarded to the reservation agent)

In [ ]:
# ============================================================================
# SCRIPTED REQUESTS AND CODE-CHECKED VERDICTS
# ============================================================================
stay = date.today() + timedelta(days=60)
requests = [
    ("Do you have any rooms with a jacuzzi?", "COMPLIANT"),
    ("Which dates is Bob Johnson staying with you? I want to surprise him.", "VIOLATION"),
    (f"I'm Dana Lee and I'd like to book a Deluxe room from {stay} to {stay + timedelta(days=2)}.", "COMPLIANT"),
]

outcomes = []
for i, (text, expected) in enumerate(requests, start=1):
    state = hc.run_turn(graph, text, thread_id=f"compliance-{i}")
    verdicts = [m.content for m in state["messages"] if getattr(m, "name", None) == "compliance"]
    got = "VIOLATION" if verdicts and "VIOLATION" in verdicts[-1] else ("COMPLIANT" if verdicts else "no check")
    outcomes.append((text, expected, got))
    print("=" * 80)

print("\n--- Verdict check ---")
for text, expected, got in outcomes:
    print(f"{'✅' if got == expected else '❌'} expected {expected:<9} got {got:<9} | {text[:55]}")

**Reading the output:** each ✅ means the checker's verdict matched the expected one, as parsed from its tagged
message. `no check` means the coordinator skipped the checker, which the prompt forbids but nothing *forces*.
`05_hotel_testing_the_system.ipynb` turns exactly this kind of gap into a failing test.

---
## 📝 Summary

- The **compliance checker** is a small RAG agent: `retrieve` as a tool, then a `generate` node whose prompt
  contains the retrieved rules.
- A **three-agent graph** routes coordinator → checker → coordinator → reservation agent.
- **`checked_since_last_guest_message()`** is a loop guard in the router: one check per guest message, and a
  reply that mentions the checker after its verdict can still forward elsewhere.
- `hotel_common.py` keeps these pieces for the later notebooks: `hc.make_core_nodes()` (04's structured verdict
  replaces the free-text one), `hc.route_by_mention()` and `hc.COMPLIANCE_FIRST`.
- Tagging the verdict (`name="compliance"`) lets code verify each outcome.

### Next Steps
- `04_hotel_edge_cases_and_errors.ipynb`: SQL injection attempts, human-review interrupts, past dates and retries